In [1]:
# Cell 1：检查环境
import subprocess
result = subprocess.run(
    ["pip", "install", "-q", "transformers", "bitsandbytes", "datasets", "accelerate"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "installed")
print(result.stderr[-300:] if result.stderr else "")

import torch
import transformers
import bitsandbytes as bnb

print(f"\ntorch:          {torch.__version__}")
print(f"transformers:   {transformers.__version__}")
print(f"bitsandbytes:   {bnb.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00



torch:          2.10.0+cu128
transformers:   5.0.0
bitsandbytes:   0.49.2
CUDA available: True
GPU:            Tesla T4
VRAM:           15.6 GB


In [2]:
# Cell 2：加载 tokenizer + 准备评估数据
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 加载 WikiText-2 测试集
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# 拼接所有文本，然后按 1024 token 分块
# GPT-2 最大上下文是 1024，超出会截断
full_text = "\n\n".join(dataset["text"])
encodings = tokenizer(full_text, return_tensors="pt")

total_tokens = encodings.input_ids.shape[1]
print(f"WikiText-2 test set 总 token 数: {total_tokens:,}")
print(f"按 1024 分块后共 {total_tokens // 1024} 个 chunk")
print(f"tokenizer vocab size: {tokenizer.vocab_size:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (287644 > 1024). Running this sequence through the model will result in indexing errors


WikiText-2 test set 总 token 数: 287,644
按 1024 分块后共 280 个 chunk
tokenizer vocab size: 50,257


In [3]:
# Cell 3：perplexity 计算函数
import torch
from tqdm import tqdm

def compute_perplexity(model, encodings, device, chunk_size=1024, max_chunks=50):
    """
    计算模型在 encodings 上的 perplexity。

    - chunk_size=1024：GPT-2 最大上下文长度
    - max_chunks=50：只取前 50 个 chunk，够稳定又不用跑太久
    """
    model.eval()
    total_nll = 0.0   # 累计 negative log likelihood
    total_tokens = 0

    input_ids = encodings.input_ids.to(device)
    n_chunks = min(max_chunks, input_ids.shape[1] // chunk_size)

    with torch.no_grad():
        for i in tqdm(range(n_chunks), desc="computing PPL"):
            start = i * chunk_size
            end   = start + chunk_size
            chunk = input_ids[:, start:end]   # shape: (1, 1024)

            # labels = input_ids 本身，模型内部做 shift（预测下一个 token）
            outputs = model(chunk, labels=chunk)

            # outputs.loss 是当前 chunk 的平均 NLL（per token）
            nll = outputs.loss.item()
            total_nll    += nll * chunk_size
            total_tokens += chunk_size

    avg_nll = total_nll / total_tokens
    ppl = torch.exp(torch.tensor(avg_nll)).item()
    return ppl

print("函数定义完成，准备加载模型")
print(f"将使用前 50 个 chunk（{50 * 1024:,} tokens）计算 PPL")

函数定义完成，准备加载模型
将使用前 50 个 chunk（51,200 tokens）计算 PPL


In [4]:
# Cell 4：FP32 baseline PPL
import time
from transformers import AutoModelForCausalLM

device = "cuda"

print("加载 GPT-2 FP32...")
model_fp32 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to(device)

# 看一下实际占用多少显存
mem_fp32 = torch.cuda.memory_allocated() / 1e9
print(f"显存占用: {mem_fp32:.2f} GB")
print(f"参数量:   {sum(p.numel() for p in model_fp32.parameters()):,}")

# 测 PPL
t0 = time.time()
ppl_fp32 = compute_perplexity(model_fp32, encodings, device)
t1 = time.time()

print(f"\nFP32 PPL:  {ppl_fp32:.2f}")
print(f"耗时:      {t1 - t0:.1f}s")

加载 GPT-2 FP32...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

显存占用: 0.51 GB
参数量:   124,439,808


computing PPL: 100%|██████████| 50/50 [00:04<00:00, 10.94it/s]


FP32 PPL:  32.35
耗时:      4.6s


In [ ]:
# Cell 5：INT8 量化 PPL（transformers 5.0 写法）
import gc
from transformers import BitsAndBytesConfig

# 先释放 FP32 模型显存
del model_fp32
gc.collect()
torch.cuda.empty_cache()

In [7]:
print(f"释放后显存: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# transformers 5.0：量化参数必须通过 BitsAndBytesConfig 传
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

print("\n加载 GPT-2 INT8...")
model_int8 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

mem_int8 = torch.cuda.memory_allocated() / 1e9
print(f"显存占用: {mem_int8:.2f} GB")

# 测 PPL
t0 = time.time()
ppl_int8 = compute_perplexity(model_int8, encodings, device)
t1 = time.time()

print(f"\nINT8 PPL:  {ppl_int8:.2f}")
print(f"耗时:      {t1 - t0:.1f}s")

释放后显存: 0.01 GB

加载 GPT-2 INT8...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


显存占用: 0.27 GB


computing PPL:   0%|          | 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
computing PPL: 100%|██████████| 50/50 [00:04<00:00, 10.28it/s]


INT8 PPL:  32.50
耗时:      4.9s


In [8]:
# Cell 6：汇总打印
print("=" * 40)
print("      Week 26 量化实验结果汇总")
print("=" * 40)
print(f"{'':12} {'显存':>8} {'PPL':>8} {'耗时':>8}")
print("-" * 40)
print(f"{'FP32':12} {'0.51 GB':>8} {ppl_fp32:>8.2f} {'4.6s':>8}")
print(f"{'INT8':12} {'0.27 GB':>8} {ppl_int8:>8.2f} {'4.9s':>8}")
print("-" * 40)
print(f"{'精度损失':12} {'':>8} {ppl_int8 - ppl_fp32:>+8.2f} {'':>8}")
print(f"{'显存节省':12} {'47%':>8}")
print("=" * 40)

      Week 26 量化实验结果汇总
                   显存      PPL       耗时
----------------------------------------
FP32          0.51 GB    32.35     4.6s
INT8          0.27 GB    32.50     4.9s
----------------------------------------
精度损失                     +0.15         
显存节省              47%
